# Lab 1: State & Nodes (Applied Version)
 building  a basic **Smart Content Generation System**.
 application takes a topic, generates a draft blog post, and measures its word count.

We will use the following concepts:
- **State**: Tracking `topic`, `draft`, `word_count`, and `status`.
- **Nodes**: Defining `generate_draft_node` (generating text using LLM) and `add_metadata_node` (processing statistics).

In [1]:
import os
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Verify API keys
print("OpenAI API Key set:", "OPENAI_API_KEY" in os.environ)
print("LangSmith tracing set:", os.environ.get("LANGCHAIN_TRACING_V2"))

OpenAI API Key set: True
LangSmith tracing set: true


### 1. Define State and Nodes


In [2]:
from typing import TypedDict
from mock_llm import get_llm
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

# State definitions
class ContentState(TypedDict):
    topic: str
    draft: str
    word_count: int
    status: str

llm = get_llm(model="gpt-4o-mini", temperature=0.7)

# Node 1: Generate Draft
def generate_draft_node(state: ContentState):
    print("--- Node: Generating Draft ---")
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a professional copywriter. Write a brief blog post paragraph (under 100 words) about the provided topic."),
        ("human", "Topic: {topic}")
    ])
    chain = prompt | llm
    response = chain.invoke({"topic": state["topic"]})
    return {
        "draft": response.content.strip(),
        "status": "drafted"
    }

# Node 2: Add Metadata
def add_metadata_node(state: ContentState):
    print("--- Node: Adding Metadata ---")
    words = len(state["draft"].split())
    return {
        "word_count": words,
        "status": "analyzed"
    }

--- OpenAI API connection failed (Error code: 429 - {'error': {'message': 'You exceeded your c...). Falling back to Mock LLM ---


### 2. Build and Compile the Graph
Now  define the sequential flow: `START -> generate_draft -> add_metadata -> END`.

In [3]:
builder = StateGraph(ContentState)
builder.add_node("generate_draft", generate_draft_node)
builder.add_node("add_metadata", add_metadata_node)

builder.add_edge(START, "generate_draft")
builder.add_edge("generate_draft", "add_metadata")
builder.add_edge("add_metadata", END)

graph = builder.compile()

try:
    graph.get_graph().print_ascii()
except Exception as e:
    print("Could not draw graph:", e)

  +-----------+    
  | __start__ |    
  +-----------+    
         *         
         *         
         *         
+----------------+ 
| generate_draft | 
+----------------+ 
         *         
         *         
         *         
 +--------------+  
 | add_metadata |  
 +--------------+  
         *         
         *         
         *         
    +---------+    
    | __end__ |    
    +---------+    


### 3. Run the Generation Pipeline

In [4]:
initial_input = {
    "topic": "The benefits of learning asynchronous programming",
    "draft": "",
    "word_count": 0,
    "status": "pending"
}

final_state = graph.invoke(initial_input)

print("\n--- Execution Output ---")
print("Draft Status:", final_state["status"])
print("Word Count:", final_state["word_count"])
print("Draft Text:\n", final_state["draft"])

--- Node: Generating Draft ---
--- Node: Adding Metadata ---

--- Execution Output ---
Draft Status: analyzed
Word Count: 37
Draft Text:
 Learning asynchronous programming allows developers to write highly concurrent applications. By managing tasks without blocking execution threads, programs run faster and handle massive user traffic efficiently. As a result, systems become responsive and modern web tools thrive.
